# Praveen Chatbot Lab

## Overview
In this lab, we will be creating an intelligent chatbot designed to answer any questions related to Praveen. This chatbot will serve as a personalized assistant that can provide information, insights, and responses about Praveen's background, expertise, projects, and other relevant topics.

## Objectives
- Build a conversational AI agent focused on Praveen-related queries
- Implement natural language processing capabilities
- Create a user-friendly interface for seamless interaction
- Ensure accurate and helpful responses to user questions

Let's get started building this Praveen-focused chatbot!

In [7]:
# Loading the python packages

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader

# It's python library to build user interfaces
# Gradio is a library that allows you to quickly create UIs for machine learning models.
import gradio as gr
import os

In [12]:
# Parse the .eve file and load the environment variables
load_dotenv(override=True)

# Get the OpenAI API key from the environment variables
openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key:
    print(f"OpenAI API key loaded successfully: {openai_api_key[:4]}...")  # Print only the first 4 characters for security
else:
    print("OpenAI API key not found. Please check your .env file.")

OpenAI API key loaded successfully: sk-p...


In [13]:
# Create an openai instance
# If the OPENAI_API_KEY is set in the environment, it will be used automatically.
openai = OpenAI()

In [14]:
# Reading the PDF file

reader = PdfReader("pramlinkedin.pdf")
linkedin = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [15]:
print(linkedin)

   
Contact
praveen.m144@gmail.com
www.linkedin.com/in/praveen-
mariyappa-a3185444 (LinkedIn)
Top Skills
Microsoft Technologies
SQL
Debugging
Praveen Mariyappa
Sr Beta Engineer at Microsoft
Redmond, Washington, United States
Summary
IT professional with over several years of experience across a
range of industries. Extensive knowledge in Microsoft platforms,
cloud technologies with deep interest in Azure, distributed systems
and Big Data. Exceptional problem solving skills combined with
strong attention to detail allowing for rapid solution delivery, efficient
operation, and accurate metrics
Experience
Microsoft
19 years 8 months
Sr Beta Engineer (Microsoft Azure)
September 2021 - Present (3 years 11 months)
Redmond, Washington, United States
Senior Software Engineer - Azure Storage
August 2020 - September 2021 (1 year 2 months)
Redmond, Washington, United States
Sr Program Manager
March 2018 - August 2020 (2 years 6 months)
Redmond, Washington, United States
Sr Embedded Escalation Eng

In [16]:
# Open the summary.txt file and read its content
with open("summary.txt", "r", encoding="utf-8") as file:
    summary = file.read()

In [17]:
print(summary)

Praveen Mariyappa is an experienced technology professional with deep expertise in Azure cloud services, particularly in Azure Kubernetes Service (AKS), Azure Container Registry (ACR), and Azure Container Instances (ACI). As a Beta Engineer, he plays a pivotal role in onboarding new feature releases, ensuring support readiness, and improving serviceability across Azure. Praveen has led the successful onboarding of over 68 feature releases, collaborated with multiple cross-functional teams, and contributed to critical support assets such as self-help content, troubleshooting guides, scoping questions, and internal documentation.

Praveen is passionate about enabling support engineers and customers to resolve issues faster through proactive measures. His contributions have significantly improved case deflection rates and reduced ticket volumes, especially within AKS and Azure Container Storage. He actively participates in bug bashes, identifies product bugs early, and raises supportabili

In [11]:
name = "Praveen Mariyappa"

In [23]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [24]:
system_prompt

"You are acting as Praveen Mariyappa. You are answering questions on Praveen Mariyappa's website, particularly questions related to Praveen Mariyappa's career, background, skills and experience. Your responsibility is to represent Praveen Mariyappa for interactions on the website as faithfully as possible. You are given a summary of Praveen Mariyappa's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nPraveen Mariyappa is an experienced technology professional with deep expertise in Azure cloud services, particularly in Azure Kubernetes Service (AKS), Azure Container Registry (ACR), and Azure Container Instances (ACI). As a Beta Engineer, he plays a pivotal role in onboarding new feature releases, ensuring support readiness, and improving serviceability across Azure. Praveen has led the successful on

In [5]:
# Creating a chat function
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    return response.choices[0].message.content

In [8]:
# Or launch in a way that you can close it programmatically
demo = gr.ChatInterface(chat, type="messages")
demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [9]:
demo.close()  # Close the interface programmatically if needed

Closing server running on port: 7861


I will use a Pydantic Model to evaluate the answer provided by the chatbot.

In [10]:
# Create a pydantic model for the chat interface
# This is not necessary for the basic functionality, but can be useful for validation and type checking

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [18]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [19]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [20]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [21]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [25]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [26]:
reply

'I do not currently hold a patent. My focus has primarily been on contributing to serviceability, support readiness, and feature releases within Azure, rather than patent development. However, I am always open to innovative ideas and collaboration that could lead to new solutions in the tech space. If you have any specific innovations in mind, feel free to share!'

In [27]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback='The response is acceptable. It clearly answers the question, provides a reason for not having a patent that aligns with the provided background, and ends with an engaging open-ended question.')

In [28]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [29]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
# Or launch in a way that you can close it programmatically
demo = gr.ChatInterface(chat, type="messages")
demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Passed evaluation - returning reply


In [33]:
demo.close()  # Close the interface programmatically if needed

Closing server running on port: 7861
